# 1 - Fine-tune YOLOv8 for the traffic observer

Fine-tunes **YOLOv8s** on **VisDrone2019-DET** (elevated / aerial traffic scenes, ~6.5k training images) so the
project sees **small, far-away and high-angle vehicles and pedestrians** that the stock COCO model misses.

The classes are remapped to the project's own set and keep the **COCO ids** the project's `Detector` expects
(`person=0, bicycle=1, car=2, motorcycle=3, bus=5, truck=7`), so the result is a **drop-in replacement**:

```
python -m traffic_watch --source clip.mp4 --weights yolov8s_traffic_best.pt --imgsz 960
```

**Honest limits.** VisDrone is drone footage. Expect gains on elevated / far views, and **possibly a loss on
dashcam views** - always compare on your own clips. VisDrone has no licence file; it is for research use
(cite: *Detection and Tracking Meet Drones Challenge*, IEEE TPAMI 2022).

**Time:** about 4-6 h for 30 epochs on a T4. It **stops itself before Kaggle's 12 h limit** (an 11 h deadline,
minus a reserve for the final evaluation) and keeps a copy of `last.pt` in the outputs after every epoch, so a run that
is cut off can be continued with `RESUME_FROM`.
Set `QUICK = True` first to check everything works (~10 min).

### How to run on Kaggle
1. **Settings (right panel):** Accelerator = **GPU T4 x2** (or P100), **Internet = On**.
2. **Run All** to test, or **Save Version -> Save & Run All (Commit)** so the outputs survive after the session.
3. Everything you need to keep is written to `/kaggle/working/outputs/` (also zipped at the end). Download it from the notebook's **Output** tab.


In [ ]:
!pip install -q -U ultralytics
import os, sys, json, time, random, re, glob, math, shutil
from pathlib import Path

ROOT = Path(os.environ.get("WORK", "/kaggle/working"))   # Kaggle's writable folder
OUT = ROOT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)

# Kaggle stops a session after 12 h. T0 is the clock; every long step checks hours_left().
T0 = time.time()
HARD_LIMIT_H = 11.0          # leave a 1 h margin below Kaggle's 12 h


def hours_left(reserve_h=0.0):
    """Hours still available before the HARD_LIMIT_H deadline, minus a reserve for later steps."""
    return HARD_LIMIT_H - (time.time() - T0) / 3600 - reserve_h

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable a GPU accelerator")


In [ ]:
# ---------------- settings ----------------
QUICK = False            # True: 1 epoch on 10% of the data, just to check the pipeline
BASE = "yolov8s.pt"      # starting weights (COCO-pretrained, downloaded automatically)
IMGSZ = 960              # the project runs at 960; larger = better for small objects, slower
EPOCHS = 1 if QUICK else 30
BATCH = 8                # lower to 4 if you hit an out-of-memory error
AMP = True
RESERVE_H = 0.7          # hours kept free for the final evaluation, saving and zipping
RESUME_FROM = None       # ONLY for a run that was cut off (crash / disconnect): the last.pt copy in that
                         # run's outputs, e.g. "/kaggle/input/<earlier-output>/last.pt".
                         # A run that finished or stopped on the time guard cannot be resumed: to train
                         # further, set BASE to its best.pt and start a new run.
FRAC = 0.10 if QUICK else 1.0


In [ ]:
# ---------------- download VisDrone (about 2 GB, a few minutes) ----------------
from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

settings.update({"datasets_dir": str(ROOT / "datasets")})
info = check_det_dataset("VisDrone.yaml")        # downloads + converts annotations to YOLO format
train_images, val_images = Path(info["train"]), Path(info["val"])
print("train images:", train_images, "\nval images  :", val_images)


In [ ]:
# ---------------- remap VisDrone classes -> the project's classes ----------------
# VisDrone: 0 pedestrian, 1 people, 2 bicycle, 3 car, 4 van, 5 truck, 6 tricycle,
#           7 awning-tricycle, 8 bus, 9 motor
# Target ids are the COCO ids the project's Detector uses.
MAP = {0: 0, 1: 0,        # pedestrian, people      -> person
       2: 1,              # bicycle                  -> bicycle
       3: 2, 4: 2,        # car, van                 -> car
       9: 3,              # motor                    -> motorcycle
       8: 5,              # bus                      -> bus
       5: 7,              # truck                    -> truck
       6: 2, 7: 2}        # tricycles                -> car (a motorised 3-wheeler; change if you disagree)
NAMES = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle",
         4: "unused4", 5: "bus", 6: "unused6", 7: "truck"}   # 4 and 6 keep the COCO ids aligned

DST = ROOT / "traffic_yolo"


from ultralytics.data.utils import img2label_paths


def build_split(img_dir, split, frac=1.0):
    img_dir = Path(img_dir)
    out_i, out_l = DST / "images" / split, DST / "labels" / split
    out_i.mkdir(parents=True, exist_ok=True)
    out_l.mkdir(parents=True, exist_ok=True)
    imgs = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    if frac < 1:
        random.Random(0).shuffle(imgs)
        imgs = imgs[:max(8, int(len(imgs) * frac))]
    n_obj = n_with = n_nolabelfile = 0
    for p in imgs:
        dst = out_i / p.name
        if not dst.exists():
            try:
                os.link(p, dst)            # hard link: no extra disk space
            except OSError:
                shutil.copy2(p, dst)
        rows = []
        # the SAME rule YOLO uses to find an image's label (".../images/..." -> ".../labels/..."),
        # so it works for both layouts of the VisDrone download
        lf = Path(img2label_paths([str(p)])[0])
        if lf.exists():
            for line in lf.read_text().splitlines():
                parts = line.split()
                if len(parts) == 5 and int(parts[0]) in MAP:
                    rows.append(" ".join([str(MAP[int(parts[0])])] + parts[1:]))
        else:
            n_nolabelfile += 1
        (out_l / (p.stem + ".txt")).write_text("\n".join(rows))
        n_obj += len(rows)
        n_with += bool(rows)
    return len(imgs), n_obj, n_with, n_nolabelfile


for split, d in (("train", train_images), ("val", val_images)):
    n_img, n_obj, n_with, n_nolabel = build_split(
        d, split, FRAC if split == "train" else max(FRAC, 0.25 if QUICK else 1.0))
    print(f"{split}: {n_img} images | {n_with} have boxes | {n_obj} boxes | {n_nolabel} images had no label file")
    if n_obj == 0 or n_with < 0.5 * n_img:
        print("\nfolders found under", info["path"], ":")
        for q in sorted(x for x in Path(info["path"]).rglob("*") if x.is_dir())[:40]:
            print("  ", q)
        raise RuntimeError(
            f"No usable labels for the '{split}' split ({n_obj} boxes in {n_img} images). Training on this "
            f"would learn nothing - stop here. The folder list above shows where the VisDrone download put "
            f"its files; the label files must sit in the 'labels' folder that mirrors 'images'.")

for c in DST.rglob("*.cache"):         # YOLO label caches left by an earlier (possibly broken) run
    c.unlink()

(DST / "data.yaml").write_text(
    f"path: {DST}\ntrain: images/train\nval: images/val\nnames:\n"
    + "\n".join(f"  {k}: {v}" for k, v in NAMES.items()) + "\n")
print((DST / "data.yaml").read_text())


In [ ]:
# ---------------- baseline: the stock COCO model on the same validation set ----------------
# (so the gain from fine-tuning is measured, not assumed)
KEEP = [0, 1, 2, 3, 5, 7]
results = {"settings": dict(base=BASE, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, quick=QUICK)}
try:
    m0 = YOLO(BASE).val(data=str(DST / "data.yaml"), imgsz=IMGSZ, batch=BATCH, classes=KEEP,
                        workers=2, plots=False, verbose=False)
    results["baseline_coco"] = dict(map50=float(m0.box.map50), map50_95=float(m0.box.map))
    print("baseline  mAP50 %.3f  mAP50-95 %.3f" % (m0.box.map50, m0.box.map))
except Exception as e:
    print("baseline evaluation skipped:", type(e).__name__, e)


In [ ]:
# ---------------- train (stops itself before Kaggle's time limit) ----------------
print(f"{hours_left():.1f} h left on the session clock; {RESERVE_H} h are reserved for the final steps")


def watch_clock(trainer):
    # runs after every epoch, once the weights are saved. (Ultralytics' own `time=` argument is not
    # used: it rewrites the epoch count so training fills the WHOLE budget instead of just capping it.)
    done = trainer.epoch - trainer.start_epoch + 1
    per_epoch_h = (time.time() - trainer.train_time_start) / 3600 / done
    if Path(trainer.last).exists():
        shutil.copy2(trainer.last, OUT / "last.pt")            # survives if the session dies
    if hours_left(RESERVE_H) < 1.15 * per_epoch_h and not trainer.stop:
        print(f"[time guard] {hours_left():.1f} h left and one epoch takes {per_epoch_h:.2f} h "
              f"-> stopping after epoch {trainer.epoch + 1}")
        trainer.stop = True


if RESUME_FROM:                                                # continue a run that was cut off
    ck = torch.load(RESUME_FROM, map_location="cpu", weights_only=False)
    assert ck.get("optimizer") is not None and ck.get("epoch", -1) >= 0, (
        "RESUME_FROM must be the last.pt of a run that was CUT OFF. A finished run (or one stopped by the "
        "time guard) cannot be resumed - set BASE to its best.pt and start a new run instead.")
    model = YOLO(RESUME_FROM)
    model.add_callback("on_fit_epoch_end", watch_clock)
    model.train(resume=True)
else:
    model = YOLO(BASE)
    model.add_callback("on_fit_epoch_end", watch_clock)
    model.train(data=str(DST / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
                workers=2, project=str(ROOT / "runs"), name="traffic_yolo", exist_ok=True,
                patience=10, cos_lr=True, close_mosaic=3, amp=AMP, seed=0, plots=True)
assert "traffic_yolo" in str(model.trainer.args.data), (
    f"trained on the wrong dataset ({model.trainer.args.data}) - do not use these weights")
best = Path(model.trainer.best)
print("best weights:", best, "| finished with", f"{hours_left():.1f} h left")


In [ ]:
# ---------------- evaluate the fine-tuned model and save everything ----------------
ft = YOLO(str(best))
m1 = ft.val(data=str(DST / "data.yaml"), imgsz=IMGSZ, batch=BATCH, workers=2, plots=True, verbose=False)
results["finetuned"] = dict(map50=float(m1.box.map50), map50_95=float(m1.box.map))
per_class = {}
for i, ap in zip(m1.box.ap_class_index, m1.box.maps[m1.box.ap_class_index]):
    per_class[ft.names[int(i)]] = round(float(ap), 3)
results["finetuned_per_class_map50_95"] = per_class
print("fine-tuned mAP50 %.3f  mAP50-95 %.3f" % (m1.box.map50, m1.box.map))
print("per class:", per_class)

shutil.copy2(best, OUT / "yolov8s_traffic_best.pt")
for f in ("results.csv", "results.png", "confusion_matrix.png", "PR_curve.png"):
    src = Path(model.trainer.save_dir) / f
    if src.exists():
        shutil.copy2(src, OUT / f)
(OUT / "yolo_results.json").write_text(json.dumps(results, indent=2))
(OUT / "HOW_TO_USE_YOLO.txt").write_text(
    "Copy yolov8s_traffic_best.pt next to the project and run:\n"
    "  python -m traffic_watch --source clip.mp4 --weights yolov8s_traffic_best.pt --imgsz 960\n"
    "Class ids match COCO (person 0, bicycle 1, car 2, motorcycle 3, bus 5, truck 7), so no code change is needed.\n"
    "Compare against the stock yolov8s.pt on YOUR clips before adopting it: VisDrone is aerial footage.\n")
print(json.dumps(results, indent=2))


In [ ]:
# ---------------- zip the outputs so they download in one click ----------------
shutil.make_archive(str(ROOT / "yolo_outputs"), "zip", OUT)
print("Download:", ROOT / "yolo_outputs.zip", "- contains", sorted(p.name for p in OUT.iterdir()))
